# Context Engineering: Hands-On Notebook

**Companion notebook for Article 1: "Understanding Context Engineering"**

This notebook provides runnable code for the concepts explained in the article. Run the cells in order.

---

## Setup

First, let's install and import the required packages.

In [19]:
# Install required packages if not already installed
!pip3 install google-adk python-dotenv aiosqlite -q

In [20]:
import os
from pathlib import Path
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Load environment variables from .env file
# This looks for .env in the project root (parent of notebooks/)
env_path = Path(__file__).parent.parent / ".env" if "__file__" in globals() else Path("../.env")
load_dotenv(dotenv_path=env_path)

# Get API key from environment
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found! Please:\n"
        "1. Copy .env.example to .env\n"
        "2. Add your API key to .env\n"
        "3. Get your key from: https://aistudio.google.com/apikey"
    )

os.environ["GOOGLE_API_KEY"] = api_key

# Initialize the client
client = genai.Client()
MODEL_ID = "gemini-2.5-flash"

print("✅ Environment loaded successfully")

✅ Environment loaded successfully


## 1. The Stateless Problem

See why LLMs forget everything between calls.

In [21]:
def stateless_call(message: str) -> str:
    """Make a stateless API call - no conversation history."""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=message
    )
    return response.text

In [22]:
# First message - introduce ourselves
response1 = stateless_call("My name is Alex and I'm a software engineer who loves hiking.")
print("User: My name is Alex and I'm a software engineer who loves hiking.")
print(f"Agent: {response1}")

User: My name is Alex and I'm a software engineer who loves hiking.
Agent: Hey Alex! Nice to meet you.

That's a fantastic combination – a software engineer who loves hiking. I imagine it offers a great balance between deep mental work and refreshing physical activity.

*   **To acknowledge and open a conversation:**
    *   "That's a great intro, Alex! I can totally see how hiking would be a perfect way to decompress and recharge after diving deep into code. Do you have any favorite trails or regions you love to explore?"
    *   "Hey Alex! Software engineering and hiking – a wonderful blend of precision and exploration. What kind of software do you usually work on?"
    *   "Nice to meet you, Alex! That sounds like a wonderful balance. Do you find any interesting crossovers between problem-solving in code and problem-solving on the trail?"

Which kind of response are you looking for? Or was that just an introduction?


In [23]:
# Second message - ask about what we just said
response2 = stateless_call("What's my name and what do I do for work?")
print("User: What's my name and what do I do for work?")
print(f"Agent: {response2}")

User: What's my name and what do I do for work?
Agent: I'm sorry, but as an AI, I don't have access to any personal information about you, including your name or what you do for work. I don't retain memory of past conversations or have any way to identify individual users.

If you'd like to tell me, I'd be happy to know!


**Result:** The agent can't answer basic questions about information provided seconds earlier.

---
## 2. ADK Sessions: The Solution

Sessions maintain conversation context automatically.

In [24]:
from google.adk.agents import LlmAgent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

# Create agent
agent = LlmAgent(
    model=MODEL_ID,
    name="TravelAgent",
    instruction="You are a helpful travel assistant. Be concise."
)
APP_NAME = "context_engineering_demo"
# Create session service and session
session_service = InMemorySessionService()
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id="user_123"
)

print(f"Session created: {session.id}")

Session created: c1c52b30-e07d-4c37-ab6b-a829bce19741


In [25]:
# Create runner and chat function
runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)

async def chat(user_message: str):
    """Send a message and get a response."""
    content = types.Content(role="user", parts=[types.Part(text=user_message)])
    
    async for event in runner.run_async(user_id="user_123", session_id=session.id, new_message=content):
        if event.is_final_response() and event.content and event.content.parts:
            return event.content.parts[0].text
    
    return "[No response]"

In [26]:
# Multi-turn conversation with memory!
print("Turn 1:")
print("User: My name is Sarah and I'm planning a trip to Tokyo.")
response = await chat('My name is Sarah and I am planning a trip to Tokyo.')
print(f"Agent: {response}")

Turn 1:
User: My name is Sarah and I'm planning a trip to Tokyo.
Agent: Hello Sarah! Tokyo is a wonderful choice. How can I help you plan your trip?
Agent: Hello Sarah! Tokyo is a wonderful choice. How can I help you plan your trip?


In [27]:
print("Turn 2:")
print("User: I want to stay for 2 weeks in April.")
response = await chat('I want to stay for 2 weeks in April.')
print(f"Agent: {response}")

Turn 2:
User: I want to stay for 2 weeks in April.
Agent: Great! Two weeks in April is perfect for enjoying spring in Tokyo. What kind of accommodation are you looking for?
Agent: Great! Two weeks in April is perfect for enjoying spring in Tokyo. What kind of accommodation are you looking for?


In [28]:
print("Turn 3 - Memory test:")
print("User: What's my name and where am I going?")
response = await chat('What is my name and where am I going?')
print(f"Agent: {response}")

Turn 3 - Memory test:
User: What's my name and where am I going?
Agent: Your name is Sarah, and you are going to Tokyo.
Agent: Your name is Sarah, and you are going to Tokyo.


**Result:** The agent remembers everything from the conversation.

---
## 3. Examining Session Structure

Look inside the session to see what's stored.

In [29]:
# Retrieve session and examine events
retrieved_session = await session_service.get_session(
    app_name=APP_NAME,
    user_id="user_123",
    session_id=session.id
)

print(f"Session ID: {retrieved_session.id}")
print(f"User ID: {retrieved_session.user_id}")
print(f"Total events: {len(retrieved_session.events)}")
print(f"State: {retrieved_session.state}")

Session ID: c1c52b30-e07d-4c37-ab6b-a829bce19741
User ID: user_123
Total events: 6
State: {}


In [30]:
# Show conversation history from events
print("CONVERSATION HISTORY")
print("=" * 50)
for i, event in enumerate(retrieved_session.events):
    if event.content and event.content.parts:
        role = event.content.role or "unknown"
        text = event.content.parts[0].text[:80] if event.content.parts[0].text else "[no text]"
        print(f"{i+1}. [{role}] {text}..." if len(text) >= 80 else f"{i+1}. [{role}] {text}")

CONVERSATION HISTORY
1. [user] My name is Sarah and I am planning a trip to Tokyo.
2. [model] Hello Sarah! Tokyo is a wonderful choice. How can I help you plan your trip?
3. [user] I want to stay for 2 weeks in April.
4. [model] Great! Two weeks in April is perfect for enjoying spring in Tokyo. What kind of ...
5. [user] What is my name and where am I going?
6. [model] Your name is Sarah, and you are going to Tokyo.


---
## 4. Session State: Working Memory

Store structured data that persists across turns.

In [31]:
# Create session with initial state
session_with_state = await session_service.create_session(
    app_name=APP_NAME,
    user_id="user_with_state",
    state={
        "destination": "Paris",
        "budget": 3000,
        "user:name": "Alex",      # user: prefix = persists across sessions
        "user:preferences": "window seat",
        "temp:last_search": None  # temp: prefix = discarded after invocation
    }
)

print("Session state:")
for key, value in session_with_state.state.items():
    print(f"  {key}: {value}")

Session state:
  destination: Paris
  budget: 3000
  user:name: Alex
  user:preferences: window seat


In [32]:
# Agent with state injection in instructions
stateful_agent = LlmAgent(
    model=MODEL_ID,
    name="StatefulAgent",
    instruction="""You are helping {user:name} plan their trip to {destination}.
    Their budget is ${budget}. They prefer {user:preferences}.
    Be concise and helpful."""
)

print("Agent instruction template:")
print(stateful_agent.instruction)
print()
print("→ Placeholders are replaced with state values at runtime")

Agent instruction template:
You are helping {user:name} plan their trip to {destination}.
    Their budget is ${budget}. They prefer {user:preferences}.
    Be concise and helpful.

→ Placeholders are replaced with state values at runtime


---
## 5. Updating State Correctly

Always use EventActions or context objects.

In [33]:
from google.adk.events import Event, EventActions

# CORRECT: Update state through EventActions
state_changes = {
    "booking_step": "confirmed",
    "user:loyalty_points": 1500
}

actions = EventActions(state_delta=state_changes)
event = Event(
    invocation_id="inv_123",
    author="BookingAgent",
    actions=actions
)

print("Correct way: EventActions")
print(f"   State delta: {state_changes}")
print("   → Tracked in history, persists correctly, thread-safe")

Correct way: EventActions
   State delta: {'booking_step': 'confirmed', 'user:loyalty_points': 1500}
   → Tracked in history, persists correctly, thread-safe


In [34]:
# WRONG: Direct modification
print("Wrong way: Direct modification")
print('   session.state["key"] = "value"  # Never do this!')
print("   → Not tracked, may not persist, not thread-safe")
print()
print("Inside tools/callbacks, use:")
print('   tool_context.state["key"] = "value"')
print('   callback_context.state["key"] = "value"')

Wrong way: Direct modification
   session.state["key"] = "value"  # Never do this!
   → Not tracked, may not persist, not thread-safe

Inside tools/callbacks, use:
   tool_context.state["key"] = "value"
   callback_context.state["key"] = "value"


---
## 6. Database Persistence

Use `DatabaseSessionService` for production.

## Summary

| Component | Purpose | Persistence |
|-----------|---------|-------------|
| **Session** | Conversation container | Per conversation |
| **Events** | What happened (history) | Per conversation |
| **State** | What we know (data) | Prefix-dependent |
| **InMemorySessionService** | Development | Lost on restart |
| **DatabaseSessionService** | Production | Persistent |
